# 23. RF・ExtraTrees・HGB・Logisticのモデル比較
出典: FX (2).ipynb、セルindex [50]。保存出力は results/imported_20260909/ を参照。
研究履歴です。実行順・Notebook内変数・元の価格CSVに依存し、エラーが出たコードも保存しています。
自動判定の文言は元実験の判定であり、監査済みの結論ではありません。全セル一括実行は再現手順ではありません。
[USER_HOME] は匿名化した元のパスです。元Notebook内の案内や依頼文は研究資料として保持しています。


## 元セルindex 50


In [ ]:
# ============================================================
# USDJPY ML MODEL TOURNAMENT v1
# ONE-CELL COMPLETE / HOLDOUT SAFE
# ============================================================
#
# 目的
# ------------------------------------------------------------
# 現在のRandom ForestをChampionとして、
#
# 1. Random Forest
# 2. Extra Trees
# 3. HistGradientBoosting
# 4. Logistic Regression
#
# を完全に同条件で比較する。
#
#
# 重要
# ------------------------------------------------------------
# ・15分足
# ・30分保有
# ・同じ30特徴量
# ・同じTransaction Cost
# ・同じWalk-Forward
# ・同じThreshold探索
#
# 2020～2025 OOS:
#     モデル選択期間
#
# 2026:
#     最終Holdout
#
# 2026の結果を見てモデルを選ばない。
#
# ============================================================


# ============================================================
# 0. IMPORT
# ============================================================

from pathlib import Path
from datetime import datetime
import warnings
import math

import numpy as np
import pandas as pd

from sklearn.ensemble import (
    RandomForestClassifier,
    ExtraTreesClassifier,
    HistGradientBoostingClassifier,
)

from sklearn.linear_model import LogisticRegression

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from sklearn.metrics import roc_auc_score
from sklearn.utils.class_weight import compute_sample_weight


warnings.filterwarnings("ignore")

np.random.seed(42)


# ============================================================
# 1. CONFIG
# ============================================================

SEED = 42

COST = 0.00004

THRESHOLDS = (
    0.50,
    0.52,
    0.54,
    0.55,
    0.56,
    0.58,
    0.60,
    0.62,
    0.65,
)

MIN_TRAIN_YEARS = 3

MIN_TRAIN_ROWS = 5000

MIN_EVAL_ROWS = 100

MIN_VALIDATION_TRADES = 100

BOOTSTRAP_ITERATIONS = 3000

BOOTSTRAP_BLOCK_DAYS = 20


# ============================================================
# 2. CURRENT FEATURE SET
# ============================================================

FEATURES = [

    "return_1",
    "return_2",
    "return_4",
    "return_8",
    "return_16",

    "vol_4",
    "vol_8",
    "vol_16",
    "vol_32",

    "ma5_distance",
    "ma5_slope",

    "ma10_distance",
    "ma10_slope",

    "ma20_distance",
    "ma20_slope",

    "ma50_distance",
    "ma50_slope",

    "ma100_distance",
    "ma100_slope",

    "body",
    "upper_wick",
    "lower_wick",
    "range_pct",

    "rsi14",
    "atr14",

    "distance_high_16",
    "distance_low_16",

    "hour_sin",
    "hour_cos",

    "weekday",
]


# ============================================================
# 3. DATAFRAME AUTO DETECTION
# ============================================================

def find_existing_ohlc_dataframe():

    candidates = []

    for name, obj in list(globals().items()):

        if not isinstance(obj, pd.DataFrame):
            continue

        if len(obj) < 10000:
            continue

        columns = {
            str(c).lower()
            for c in obj.columns
        }

        if {
            "open",
            "high",
            "low",
            "close",
        }.issubset(columns):

            candidates.append(
                (
                    len(obj),
                    name,
                    obj,
                )
            )

    if not candidates:
        return None, None

    candidates.sort(
        reverse=True,
        key=lambda x: x[0],
    )

    _, name, df = candidates[0]

    return name, df.copy()


# ============================================================
# 4. CSV READER
# ============================================================

def normalize_ohlc_dataframe(df):

    x = df.copy()

    # ------------------------------------------
    # Columns lowercase
    # ------------------------------------------

    x.columns = [
        str(c).strip().lower()
        for c in x.columns
    ]

    # ------------------------------------------
    # Detect timestamp
    # ------------------------------------------

    timestamp_candidates = [

        "timestamp",
        "datetime",
        "date",
        "time",

    ]

    timestamp_col = None

    for c in timestamp_candidates:

        if c in x.columns:

            timestamp_col = c

            break

    if timestamp_col is not None:

        ts = pd.to_datetime(
            x[timestamp_col],
            errors="coerce",
            utc=True,
        )

        x = x.drop(
            columns=[timestamp_col]
        )

        x.index = ts

    else:

        # --------------------------------------
        # すでにDatetimeIndex
        # --------------------------------------

        if isinstance(
            x.index,
            pd.DatetimeIndex,
        ):

            x.index = pd.to_datetime(
                x.index,
                utc=True,
            )

        else:

            # ----------------------------------
            # CSVの第1列がtimestampのケース
            # ----------------------------------

            first_col = x.columns[0]

            trial = pd.to_datetime(
                x[first_col],
                errors="coerce",
                utc=True,
            )

            if trial.notna().mean() > 0.90:

                x = x.drop(
                    columns=[first_col]
                )

                x.index = trial

            else:

                # index自身を試す

                trial_index = pd.to_datetime(
                    x.index,
                    errors="coerce",
                    utc=True,
                )

                if pd.Series(
                    trial_index
                ).notna().mean() > 0.90:

                    x.index = trial_index

                else:

                    raise RuntimeError(
                        "timestampを認識できません。"
                    )

    # ------------------------------------------
    # OHLC
    # ------------------------------------------

    required = [
        "open",
        "high",
        "low",
        "close",
    ]

    missing = [
        c
        for c in required
        if c not in x.columns
    ]

    if missing:

        raise RuntimeError(
            f"OHLC不足: {missing}"
        )

    x = x[
        required
    ].copy()

    for c in required:

        x[c] = pd.to_numeric(
            x[c],
            errors="coerce",
        )

    x = x.dropna()

    x = x.loc[
        ~x.index.isna()
    ]

    x = x.loc[
        ~x.index.duplicated(
            keep="first"
        )
    ]

    x = x.sort_index()

    return x


# ============================================================
# 5. FILE AUTO DETECTION
# ============================================================

def load_bars_auto():

    print()
    print("=" * 90)
    print("DATA SOURCE DETECTION")
    print("=" * 90)

    # ========================================================
    # A. Notebook内のDataFrame
    # ========================================================

    name, existing = find_existing_ohlc_dataframe()

    if existing is not None:

        try:

            bars = normalize_ohlc_dataframe(
                existing
            )

            print(
                "Source: notebook DataFrame"
            )

            print(
                "Variable:",
                name
            )

            print(
                "Rows:",
                len(bars)
            )

            return bars

        except Exception as e:

            print(
                "Existing DataFrame skipped:",
                e
            )

    # ========================================================
    # B. 統合CSV
    # ========================================================

    home = Path.home()
    cwd = Path.cwd()

    combined_candidates = [

        cwd
        / "data"
        / "raw"
        / "usdjpy_15m_2016_2026.csv",

        cwd
        / "usdjpy_15m_2016_2026.csv",

        home
        / "fx-ml-trading"
        / "data"
        / "raw"
        / "usdjpy_15m_2016_2026.csv",

        home
        / "Documents"
        / "fx-ml-trading"
        / "data"
        / "raw"
        / "usdjpy_15m_2016_2026.csv",

    ]

    for path in combined_candidates:

        if path.exists():

            print(
                "Source:",
                path
            )

            raw = pd.read_csv(path)

            bars = normalize_ohlc_dataframe(
                raw
            )

            print(
                "Rows:",
                len(bars)
            )

            return bars

    # ========================================================
    # C. ユーザーの年別Dukascopyファイル
    # ========================================================

    yearly_dirs = [

        home
        / "dukascopy_usdjpy",

        cwd
        / "dukascopy_usdjpy",

    ]

    for directory in yearly_dirs:

        if not directory.exists():

            continue

        files = sorted(
            directory.glob(
                "usdjpy_15m_*.csv"
            )
        )

        if len(files) == 0:

            continue

        print(
            "Using yearly CSV directory:"
        )

        print(
            directory
        )

        frames = []

        for path in files:

            try:

                raw = pd.read_csv(
                    path
                )

                frame = normalize_ohlc_dataframe(
                    raw
                )

                frames.append(
                    frame
                )

                print(
                    "Loaded:",
                    path.name,
                    len(frame)
                )

            except Exception as e:

                print(
                    "Skipped:",
                    path.name,
                    e
                )

        if frames:

            bars = pd.concat(
                frames
            )

            bars = bars.loc[
                ~bars.index.duplicated(
                    keep="first"
                )
            ]

            bars = bars.sort_index()

            print()
            print(
                "Combined rows:",
                len(bars)
            )

            return bars

    # ========================================================
    # D. Failure
    # ========================================================

    raise FileNotFoundError(

        "\n15分足データを自動検出できませんでした。\n"
        "NotebookにOHLC DataFrameを読み込むか、\n"
        "C:\\Users\\<user>\\dukascopy_usdjpy\\usdjpy_15m_YYYY.csv\n"
        "を置いてください。"

    )


# ============================================================
# 6. LOAD
# ============================================================

bars = load_bars_auto()


print()
print("=" * 90)
print("RAW DATA CHECK")
print("=" * 90)

print(
    "Rows:",
    len(bars)
)

print(
    "Period:",
    bars.index.min(),
    "->",
    bars.index.max()
)


# ============================================================
# 7. RSI
# ============================================================

def calculate_rsi(
    close,
    period=14,
):

    delta = close.diff()

    gain = delta.clip(
        lower=0
    )

    loss = -delta.clip(
        upper=0
    )

    avg_gain = gain.rolling(
        period
    ).mean()

    avg_loss = loss.rolling(
        period
    ).mean()

    rs = (
        avg_gain
        /
        avg_loss.replace(
            0,
            np.nan
        )
    )

    return (
        100
        -
        100
        /
        (
            1
            +
            rs
        )
    )


# ============================================================
# 8. FEATURE ENGINEERING
# ============================================================

def make_features(
    data
):

    x = data.copy()

    # ------------------------------------------
    # Returns
    # ------------------------------------------

    for n in [
        1,
        2,
        4,
        8,
        16,
    ]:

        x[
            f"return_{n}"
        ] = (

            x["close"]
            .pct_change(n)

        )

    # ------------------------------------------
    # Volatility
    # ------------------------------------------

    for n in [
        4,
        8,
        16,
        32,
    ]:

        x[
            f"vol_{n}"
        ] = (

            x["return_1"]
            .rolling(n)
            .std()

        )

    # ------------------------------------------
    # Moving averages
    # ------------------------------------------

    for period in [
        5,
        10,
        20,
        50,
        100,
    ]:

        ma = (

            x["close"]
            .rolling(period)
            .mean()

        )

        x[
            f"ma{period}_distance"
        ] = (

            x["close"]
            /
            ma
            -
            1

        )

        x[
            f"ma{period}_slope"
        ] = (

            ma.pct_change()

        )

    # ------------------------------------------
    # Candle structure
    # ------------------------------------------

    candle_range = (

        x["high"]
        -
        x["low"]

    ).replace(
        0,
        np.nan
    )

    x["body"] = (

        x["close"]
        -
        x["open"]

    ) / candle_range

    x["upper_wick"] = (

        x["high"]

        -

        x[
            [
                "open",
                "close",
            ]
        ].max(
            axis=1
        )

    ) / candle_range

    x["lower_wick"] = (

        x[
            [
                "open",
                "close",
            ]
        ].min(
            axis=1
        )

        -

        x["low"]

    ) / candle_range

    x["range_pct"] = (

        x["high"]
        -
        x["low"]

    ) / x["close"]

    # ------------------------------------------
    # RSI
    # ------------------------------------------

    x["rsi14"] = (

        calculate_rsi(
            x["close"],
            14
        )

        /
        100

    )

    # ------------------------------------------
    # ATR
    # ------------------------------------------

    prev_close = (
        x["close"]
        .shift(1)
    )

    true_range = pd.concat(

        [

            x["high"]
            -
            x["low"],

            (
                x["high"]
                -
                prev_close
            ).abs(),

            (
                x["low"]
                -
                prev_close
            ).abs(),

        ],

        axis=1

    ).max(
        axis=1
    )

    x["atr14"] = (

        true_range
        .rolling(14)
        .mean()

        /

        x["close"]

    )

    # ------------------------------------------
    # Recent high / low
    # ------------------------------------------

    high16 = (

        x["high"]
        .rolling(16)
        .max()

    )

    low16 = (

        x["low"]
        .rolling(16)
        .min()

    )

    x["distance_high_16"] = (

        high16
        -
        x["close"]

    ) / x["close"]

    x["distance_low_16"] = (

        x["close"]
        -
        low16

    ) / x["close"]

    # ------------------------------------------
    # UTC time
    # ------------------------------------------

    hour = (

        x.index.hour

        +

        x.index.minute
        /
        60

    )

    x["hour_sin"] = np.sin(

        2
        *
        np.pi
        *
        hour
        /
        24

    )

    x["hour_cos"] = np.cos(

        2
        *
        np.pi
        *
        hour
        /
        24

    )

    x["weekday"] = (

        x.index.dayofweek
        /
        4

    )

    return x


# ============================================================
# 9. 30-MINUTE LABEL
# ============================================================

def prepare_dataset(
    bars
):

    data = make_features(
        bars
    )

    data = data.replace(
        [
            np.inf,
            -np.inf,
        ],
        np.nan
    )

    times = pd.Series(

        bars.index,

        index=bars.index

    )

    # Signal t
    #
    # Entry:
    # Open(t+1)
    #
    # Exit:
    # Close(t+2)
    #
    # = 約30分保有

    data["entry_time"] = (
        times.shift(-1)
    )

    data["label_end"] = (

        times.shift(-2)

        +

        pd.Timedelta(
            minutes=15
        )

    )

    data["entry_price"] = (

        bars["open"]
        .shift(-1)

    )

    data["exit_price"] = (

        bars["close"]
        .shift(-2)

    )

    data["future_return"] = (

        data["exit_price"]

        /

        data["entry_price"]

        -

        1

    )

    data["target"] = (

        data["future_return"]
        >
        0

    ).astype(int)

    # ------------------------------------------
    # Weekend gap等を除去
    # ------------------------------------------

    continuous = (

        (
            times.shift(-1)
            -
            times
        ).eq(
            pd.Timedelta(
                minutes=15
            )
        )

        &

        (
            times.shift(-2)
            -
            times
        ).eq(
            pd.Timedelta(
                minutes=30
            )
        )

    )

    data = data.loc[
        continuous
    ]

    data = data.dropna(

        subset=(
            FEATURES
            +
            [
                "future_return",
                "label_end",
                "entry_time",
            ]
        )

    )

    return data.copy()


data = prepare_dataset(
    bars
)


print()
print("=" * 90)
print("ML DATASET")
print("=" * 90)

print(
    "Usable rows:",
    len(data)
)

print(
    "Features:",
    len(FEATURES)
)

print(
    "Target UP rate:",
    data["target"].mean()
)

print(
    "Years:",
    sorted(
        data.index.year.unique()
    )
)


# ============================================================
# 10. PURGED ANNUAL WALK-FORWARD
# ============================================================

def annual_splits(
    data
):

    years = sorted(
        data.index.year.unique()
    )

    for year in years:

        validation_year = (
            year
            -
            1
        )

        historical_years = [

            y
            for y in years
            if y < validation_year

        ]

        if (
            len(
                historical_years
            )
            <
            MIN_TRAIN_YEARS
        ):

            continue

        if validation_year not in years:

            continue

        val_start = pd.Timestamp(

            year=validation_year,
            month=1,
            day=1,
            tz="UTC",

        )

        test_start = pd.Timestamp(

            year=year,
            month=1,
            day=1,
            tz="UTC",

        )

        test_end = pd.Timestamp(

            year=year + 1,
            month=1,
            day=1,
            tz="UTC",

        )

        # --------------------------------------
        # Train
        # --------------------------------------

        train = data.loc[

            (
                data.index
                <
                val_start
            )

            &

            (
                data["label_end"]
                <=
                val_start
            )

        ]

        # --------------------------------------
        # Validation
        # --------------------------------------

        validation = data.loc[

            (
                data.index
                >=
                val_start
            )

            &

            (
                data.index
                <
                test_start
            )

            &

            (
                data["label_end"]
                <=
                test_start
            )

        ]

        # --------------------------------------
        # Train + Validation
        # --------------------------------------

        final_train = data.loc[

            (
                data.index
                <
                test_start
            )

            &

            (
                data["label_end"]
                <=
                test_start
            )

        ]

        # --------------------------------------
        # Test
        # --------------------------------------

        test = data.loc[

            (
                data.index
                >=
                test_start
            )

            &

            (
                data.index
                <
                test_end
            )

            &

            (
                data["label_end"]
                <=
                test_end
            )

        ]

        if (
            len(train)
            <
            MIN_TRAIN_ROWS
        ):

            continue

        if (
            len(validation)
            <
            MIN_EVAL_ROWS
        ):

            continue

        if (
            len(test)
            <
            MIN_EVAL_ROWS
        ):

            continue

        yield (

            int(year),

            train,

            validation,

            final_train,

            test,

        )


splits = list(
    annual_splits(
        data
    )
)


if len(splits) < 2:

    raise RuntimeError(
        "Walk-Forward splitを十分作成できません。"
    )


available_test_years = [

    x[0]
    for x in splits

]


# ============================================================
# 11. HOLDOUT YEAR
# ============================================================

HOLDOUT_YEAR = max(
    available_test_years
)


SELECTION_YEARS = [

    y
    for y in available_test_years
    if y < HOLDOUT_YEAR

]


print()
print("=" * 90)
print("TOURNAMENT DESIGN")
print("=" * 90)

print(
    "Model-selection OOS years:",
    SELECTION_YEARS
)

print(
    "FINAL HOLDOUT YEAR:",
    HOLDOUT_YEAR
)

print(
    "IMPORTANT: Holdout is not used to choose the model."
)


# ============================================================
# 12. MODEL FACTORY
# ============================================================

MODEL_NAMES = [

    "RANDOM_FOREST",

    "EXTRA_TREES",

    "HIST_GRADIENT_BOOSTING",

    "LOGISTIC_REGRESSION",

]


def build_model(
    model_name
):

    # ========================================================
    # Current Champion
    # ========================================================

    if model_name == "RANDOM_FOREST":

        return RandomForestClassifier(

            n_estimators=250,

            max_depth=8,

            min_samples_leaf=30,

            max_features="sqrt",

            class_weight="balanced",

            random_state=SEED,

            n_jobs=-1,

        )

    # ========================================================
    # Extra Trees
    # ========================================================

    if model_name == "EXTRA_TREES":

        return ExtraTreesClassifier(

            n_estimators=250,

            max_depth=8,

            min_samples_leaf=30,

            max_features="sqrt",

            class_weight="balanced",

            random_state=SEED,

            n_jobs=-1,

        )

    # ========================================================
    # Gradient Boosting
    # ========================================================

    if model_name == "HIST_GRADIENT_BOOSTING":

        return HistGradientBoostingClassifier(

            learning_rate=0.05,

            max_iter=180,

            max_leaf_nodes=31,

            max_depth=None,

            min_samples_leaf=30,

            l2_regularization=1.0,

            early_stopping=False,

            random_state=SEED,

        )

    # ========================================================
    # Logistic Regression
    # ========================================================

    if model_name == "LOGISTIC_REGRESSION":

        return Pipeline(

            [

                (
                    "scale",
                    StandardScaler()
                ),

                (
                    "model",

                    LogisticRegression(

                        C=1.0,

                        max_iter=1000,

                        class_weight="balanced",

                        solver="lbfgs",

                        random_state=SEED,

                    )

                ),

            ]

        )

    raise ValueError(
        model_name
    )


# ============================================================
# 13. FIT MODEL
# ============================================================

def fit_model(
    model_name,
    model,
    X,
    y,
):

    # HistGBにはbalanced sample weightを与える

    if (
        model_name
        ==
        "HIST_GRADIENT_BOOSTING"
    ):

        weights = compute_sample_weight(

            class_weight="balanced",

            y=y,

        )

        model.fit(

            X,
            y,

            sample_weight=weights,

        )

    else:

        model.fit(
            X,
            y,
        )

    return model


# ============================================================
# 14. PREDICT PROBABILITY
# ============================================================

def predict_up_probability(
    model,
    X,
):

    probability = (
        model.predict_proba(X)
    )

    classes = list(
        model.classes_
    )

    if (
        0 not in classes
        or
        1 not in classes
    ):

        raise RuntimeError(
            "Both classes are required."
        )

    up_index = classes.index(
        1
    )

    return probability[
        :,
        up_index
    ]


# ============================================================
# 15. PREDICTION TABLE
# ============================================================

def make_prediction_table(
    model,
    frame,
):

    p_up = predict_up_probability(

        model,

        frame[FEATURES],

    )

    result = frame[

        [

            "entry_time",

            "label_end",

            "entry_price",

            "exit_price",

            "future_return",

            "target",

        ]

    ].copy()

    result["p_up"] = p_up

    result["confidence"] = np.maximum(

        p_up,

        1
        -
        p_up

    )

    result["direction"] = np.where(

        p_up >= 0.5,

        "BUY",

        "SELL",

    )

    result["direction_correct"] = (

        (
            p_up
            >=
            0.5
        )

        ==

        (
            frame["future_return"]
            >
            0
        )

    )

    result["gross_return"] = (

        frame["future_return"]

        *

        np.where(

            p_up >= 0.5,

            1.0,

            -1.0,

        )

    )

    return result


# ============================================================
# 16. NON-OVERLAPPING TRADE SELECTION
# ============================================================

def select_trades(
    predictions,
    threshold,
):

    candidates = predictions.loc[

        predictions["confidence"]
        >=
        threshold

    ].sort_index()

    selected = []

    next_available_entry = None

    for row in candidates.itertuples():

        if (
            next_available_entry
            is not None

            and

            row.entry_time
            <
            next_available_entry
        ):

            continue

        selected.append(
            row.Index
        )

        next_available_entry = (
            row.label_end
        )

    trades = candidates.loc[
        selected
    ].copy()

    trades["net_return"] = (

        trades["gross_return"]

        -
        COST

    )

    trades["cost"] = COST

    return trades


# ============================================================
# 17. STRATEGY STATS
# ============================================================

def strategy_stats(
    returns
):

    r = pd.Series(
        returns
    ).dropna().astype(float)

    if len(r) == 0:

        return {

            "trades": 0,

            "win_rate": np.nan,

            "avg_return": np.nan,

            "profit_factor": np.nan,

            "growth": np.nan,

            "max_dd": np.nan,

            "return_to_dd": np.nan,

        }

    wins = r.loc[
        r > 0
    ].sum()

    losses = -r.loc[
        r < 0
    ].sum()

    if losses > 0:

        pf = (
            wins
            /
            losses
        )

    elif wins > 0:

        pf = np.inf

    else:

        pf = np.nan

    equity = (

        1.0
        +
        r

    ).cumprod()

    peak = (
        equity.cummax()
    )

    dd = (

        equity
        /
        peak
        -
        1.0

    )

    max_dd = float(
        dd.min()
    )

    growth = float(

        equity.iloc[-1]
        -
        1.0

    )

    if max_dd < 0:

        return_to_dd = (

            growth
            /
            abs(
                max_dd
            )

        )

    else:

        return_to_dd = np.nan

    return {

        "trades":
            int(
                len(r)
            ),

        "win_rate":
            float(
                (
                    r > 0
                ).mean()
            ),

        "avg_return":
            float(
                r.mean()
            ),

        "profit_factor":
            float(
                pf
            ),

        "growth":
            growth,

        "max_dd":
            max_dd,

        "return_to_dd":
            float(
                return_to_dd
            ),

    }


# ============================================================
# 18. VALIDATION THRESHOLD SEARCH
# ============================================================

def choose_threshold(
    predictions
):

    rows = []

    best_threshold = None

    best_score = -np.inf

    for threshold in THRESHOLDS:

        trades = select_trades(

            predictions,

            threshold,

        )

        stats = strategy_stats(

            trades[
                "net_return"
            ]

        )

        eligible = (

            len(trades)

            >=

            MIN_VALIDATION_TRADES

        )

        if eligible:

            score = (

                stats[
                    "avg_return"
                ]

                *

                np.sqrt(
                    len(trades)
                )

            )

        else:

            score = np.nan

        rows.append({

            "threshold":
                threshold,

            "eligible":
                eligible,

            "score":
                score,

            **stats,

        })

        if (

            eligible

            and

            score
            >
            best_score

        ):

            best_score = (
                score
            )

            best_threshold = (
                threshold
            )

    return (

        best_threshold,

        pd.DataFrame(
            rows
        ),

    )


# ============================================================
# 19. SINGLE MODEL / YEAR EVALUATION
# ============================================================

def evaluate_model_split(

    model_name,

    year,

    train,

    validation,

    final_train,

    test,

):

    # ------------------------------------------
    # Validation model
    # ------------------------------------------

    model = build_model(
        model_name
    )

    model = fit_model(

        model_name,

        model,

        train[FEATURES],

        train["target"],

    )

    validation_predictions = (
        make_prediction_table(

            model,

            validation,

        )
    )

    threshold, search_table = (
        choose_threshold(

            validation_predictions

        )
    )

    if threshold is None:

        return (
            None,
            search_table,
            pd.DataFrame(),
        )

    # ------------------------------------------
    # Final train
    # ------------------------------------------

    final_model = build_model(
        model_name
    )

    final_model = fit_model(

        model_name,

        final_model,

        final_train[FEATURES],

        final_train["target"],

    )

    # ------------------------------------------
    # Untouched test
    # ------------------------------------------

    test_predictions = (
        make_prediction_table(

            final_model,

            test,

        )
    )

    test_trades = select_trades(

        test_predictions,

        threshold,

    )

    stats = strategy_stats(

        test_trades[
            "net_return"
        ]

    )

    if (
        test["target"].nunique()
        ==
        2
    ):

        auc = roc_auc_score(

            test["target"],

            test_predictions[
                "p_up"
            ],

        )

    else:

        auc = np.nan

    result = {

        "model":
            model_name,

        "test_year":
            int(year),

        "validation_year":
            int(year - 1),

        "threshold":
            float(
                threshold
            ),

        "train_rows":
            len(train),

        "validation_rows":
            len(validation),

        "test_rows":
            len(test),

        "test_auc":
            float(
                auc
            ),

        "direction_accuracy":
            float(

                test_trades[
                    "direction_correct"
                ].mean()

            )
            if len(test_trades)
            else np.nan,

        **stats,

    }

    test_trades = (
        test_trades.copy()
    )

    test_trades["model"] = (
        model_name
    )

    test_trades["test_year"] = (
        int(year)
    )

    test_trades["threshold"] = (
        threshold
    )

    return (

        result,

        search_table,

        test_trades,

    )


# ============================================================
# 20. MODEL-SELECTION TOURNAMENT
# ============================================================

print()
print("=" * 90)
print("MODEL TOURNAMENT")
print("2020-2025 OOS ONLY")
print("=" * 90)


selection_results = []

selection_trade_frames = []

validation_search_frames = []


for (

    year,

    train,

    validation,

    final_train,

    test,

) in splits:

    if year == HOLDOUT_YEAR:

        continue

    print()
    print("=" * 70)
    print(
        "TEST YEAR:",
        year
    )
    print("=" * 70)

    for model_name in MODEL_NAMES:

        print()
        print(
            "Running:",
            model_name
        )

        try:

            (
                result,

                search_table,

                test_trades,

            ) = evaluate_model_split(

                model_name,

                year,

                train,

                validation,

                final_train,

                test,

            )

        except Exception as e:

            print(
                "FAILED:",
                type(e).__name__,
                e
            )

            continue

        if result is None:

            print(
                "No eligible threshold."
            )

            continue

        selection_results.append(
            result
        )

        search_table = (
            search_table.copy()
        )

        search_table["model"] = (
            model_name
        )

        search_table["test_year"] = (
            year
        )

        validation_search_frames.append(
            search_table
        )

        selection_trade_frames.append(
            test_trades
        )

        print(
            "AUC:",
            round(
                result["test_auc"],
                4
            )
        )

        print(
            "Threshold:",
            result["threshold"]
        )

        print(
            "Trades:",
            result["trades"]
        )

        print(
            "PF:",
            round(
                result["profit_factor"],
                3
            )
        )

        print(
            "Avg Return:",
            round(
                result["avg_return"]
                *
                100,
                5
            ),
            "%"
        )

        print(
            "Max DD:",
            round(
                result["max_dd"]
                *
                100,
                3
            ),
            "%"
        )


selection_annual = pd.DataFrame(
    selection_results
)


if selection_annual.empty:

    raise RuntimeError(
        "Tournament results are empty."
    )


selection_trades = pd.concat(

    selection_trade_frames,

    axis=0,

    ignore_index=False,

)


# ============================================================
# 21. YEARLY RANK
# ============================================================

rank_metrics = [

    "test_auc",

    "profit_factor",

    "avg_return",

    "return_to_dd",

]


for metric in rank_metrics:

    selection_annual[
        f"rank_{metric}"
    ] = (

        selection_annual

        .groupby(
            "test_year"
        )[
            metric
        ]

        .rank(

            ascending=False,

            method="average",

        )

    )


selection_annual[
    "year_mean_rank"
] = (

    selection_annual[
        [
            f"rank_{m}"
            for m in rank_metrics
        ]
    ]

    .mean(
        axis=1
    )

)


# ============================================================
# 22. AGGREGATE SELECTION SUMMARY
# ============================================================

summary_rows = []


for model_name in MODEL_NAMES:

    yearly = selection_annual.loc[

        selection_annual[
            "model"
        ]
        ==
        model_name

    ].copy()

    trades_model = selection_trades.loc[

        selection_trades[
            "model"
        ]
        ==
        model_name

    ].copy()

    if len(yearly) == 0:

        continue

    aggregate_stats = strategy_stats(

        trades_model[
            "net_return"
        ]

    )

    summary_rows.append({

        "model":
            model_name,

        "evaluated_years":
            len(yearly),

        "mean_auc":
            yearly[
                "test_auc"
            ].mean(),

        "median_auc":
            yearly[
                "test_auc"
            ].median(),

        "mean_direction_accuracy":
            yearly[
                "direction_accuracy"
            ].mean(),

        "positive_return_years":
            int(
                (
                    yearly[
                        "avg_return"
                    ]
                    >
                    0
                ).sum()
            ),

        "pf_above_1_years":
            int(
                (
                    yearly[
                        "profit_factor"
                    ]
                    >
                    1
                ).sum()
            ),

        "mean_year_rank":
            yearly[
                "year_mean_rank"
            ].mean(),

        "aggregate_trades":
            aggregate_stats[
                "trades"
            ],

        "aggregate_avg_return":
            aggregate_stats[
                "avg_return"
            ],

        "aggregate_pf":
            aggregate_stats[
                "profit_factor"
            ],

        "aggregate_growth":
            aggregate_stats[
                "growth"
            ],

        "aggregate_max_dd":
            aggregate_stats[
                "max_dd"
            ],

        "aggregate_return_dd":
            aggregate_stats[
                "return_to_dd"
            ],

    })


selection_summary = pd.DataFrame(
    summary_rows
)


selection_summary = (

    selection_summary

    .sort_values(

        [

            "mean_year_rank",

            "mean_auc",

            "aggregate_pf",

        ],

        ascending=[

            True,

            False,

            False,

        ],

    )

    .reset_index(
        drop=True
    )

)


print()
print("=" * 90)
print("MODEL-SELECTION SUMMARY")
print("=" * 90)

print(

    selection_summary.to_string(
        index=False
    )

)


# ============================================================
# 23. PAIRED DAILY BOOTSTRAP VS RF
# ============================================================

def daily_return_series(
    trades
):

    if trades.empty:

        return pd.Series(
            dtype=float
        )

    temp = trades.copy()

    temp["day"] = pd.to_datetime(

        temp["entry_time"],

        utc=True

    ).dt.normalize()

    return (

        temp

        .groupby(
            "day"
        )[
            "net_return"
        ]

        .sum()

        .sort_index()

    )


def moving_block_bootstrap(

    delta,

    block_size=20,

    iterations=3000,

    seed=42,

):

    x = np.asarray(
        delta,
        dtype=float
    )

    x = x[
        np.isfinite(x)
    ]

    n = len(x)

    if n == 0:

        return {

            "observed":
                np.nan,

            "ci_low":
                np.nan,

            "ci_high":
                np.nan,

            "prob_positive":
                np.nan,

        }

    block_size = min(
        block_size,
        n
    )

    rng = np.random.default_rng(
        seed
    )

    means = np.empty(
        iterations
    )

    blocks_needed = int(
        np.ceil(
            n
            /
            block_size
        )
    )

    max_start = max(
        0,
        n
        -
        block_size
    )

    for i in range(
        iterations
    ):

        sampled = []

        for _ in range(
            blocks_needed
        ):

            if max_start == 0:

                start = 0

            else:

                start = int(
                    rng.integers(
                        0,
                        max_start + 1
                    )
                )

            sampled.append(

                x[
                    start:
                    start + block_size
                ]

            )

        sample = np.concatenate(
            sampled
        )[:n]

        means[i] = (
            sample.mean()
        )

    return {

        "observed":
            float(
                x.mean()
            ),

        "ci_low":
            float(
                np.percentile(
                    means,
                    2.5
                )
            ),

        "ci_high":
            float(
                np.percentile(
                    means,
                    97.5
                )
            ),

        "prob_positive":
            float(
                (
                    means > 0
                ).mean()
            ),

    }


rf_selection_trades = (

    selection_trades.loc[

        selection_trades[
            "model"
        ]
        ==
        "RANDOM_FOREST"

    ]

)


rf_daily = daily_return_series(
    rf_selection_trades
)


bootstrap_rows = []


print()
print("=" * 90)
print("PAIRED BLOCK BOOTSTRAP VS RANDOM FOREST")
print("=" * 90)


for model_name in MODEL_NAMES:

    if model_name == "RANDOM_FOREST":

        continue

    challenger_trades = (

        selection_trades.loc[

            selection_trades[
                "model"
            ]
            ==
            model_name

        ]

    )

    challenger_daily = daily_return_series(
        challenger_trades
    )

    paired = pd.concat(

        [

            rf_daily.rename(
                "rf"
            ),

            challenger_daily.rename(
                "challenger"
            ),

        ],

        axis=1

    ).fillna(
        0.0
    )

    delta = (

        paired[
            "challenger"
        ]

        -

        paired[
            "rf"
        ]

    )

    boot = moving_block_bootstrap(

        delta,

        block_size=
            BOOTSTRAP_BLOCK_DAYS,

        iterations=
            BOOTSTRAP_ITERATIONS,

        seed=SEED,

    )

    bootstrap_rows.append({

        "model":
            model_name,

        **boot,

    })

    print()
    print(
        model_name
    )

    print(
        "Observed daily alpha:",
        boot["observed"]
        *
        100,
        "%"
    )

    print(
        "95% CI:",
        boot["ci_low"]
        *
        100,
        "%",
        "~",
        boot["ci_high"]
        *
        100,
        "%"
    )

    print(
        "P(alpha > 0):",
        boot["prob_positive"]
        *
        100,
        "%"
    )


bootstrap_results = pd.DataFrame(
    bootstrap_rows
)


# ============================================================
# 24. SELECT TOURNAMENT LEADER
# ============================================================

SELECTION_LEADER = str(

    selection_summary.iloc[0][
        "model"
    ]

)


print()
print("=" * 90)
print("SELECTION-PERIOD WINNER")
print("=" * 90)

print(
    "Leader:",
    SELECTION_LEADER
)

print(
    "Current Champion:",
    "RANDOM_FOREST"
)


# ============================================================
# 25. FINAL HOLDOUT SPLIT
# ============================================================

holdout_split = None


for item in splits:

    if item[0] == HOLDOUT_YEAR:

        holdout_split = item

        break


if holdout_split is None:

    raise RuntimeError(
        "Holdout split not found."
    )


(

    holdout_year,

    holdout_train,

    holdout_validation,

    holdout_final_train,

    holdout_test,

) = holdout_split


# ============================================================
# 26. FINAL HOLDOUT
#
# ここではRFとSelection Winnerだけを見る
# ============================================================

print()
print("=" * 90)
print("FINAL HOLDOUT")
print(
    "YEAR:",
    HOLDOUT_YEAR
)
print("=" * 90)


holdout_models = [

    "RANDOM_FOREST",

]


if (
    SELECTION_LEADER
    !=
    "RANDOM_FOREST"
):

    holdout_models.append(
        SELECTION_LEADER
    )


holdout_results_list = []

holdout_trade_frames = []


for model_name in holdout_models:

    print()
    print(
        "Testing:",
        model_name
    )

    (

        result,

        search_table,

        trades_result,

    ) = evaluate_model_split(

        model_name,

        holdout_year,

        holdout_train,

        holdout_validation,

        holdout_final_train,

        holdout_test,

    )

    if result is None:

        print(
            "No valid Holdout result."
        )

        continue

    holdout_results_list.append(
        result
    )

    holdout_trade_frames.append(
        trades_result
    )

    print(
        "AUC:",
        result["test_auc"]
    )

    print(
        "Threshold:",
        result["threshold"]
    )

    print(
        "Trades:",
        result["trades"]
    )

    print(
        "Accuracy:",
        result[
            "direction_accuracy"
        ]
        *
        100,
        "%"
    )

    print(
        "PF:",
        result[
            "profit_factor"
        ]
    )

    print(
        "Avg Return:",
        result[
            "avg_return"
        ]
        *
        100,
        "%"
    )

    print(
        "Growth:",
        result[
            "growth"
        ]
        *
        100,
        "%"
    )

    print(
        "Max DD:",
        result[
            "max_dd"
        ]
        *
        100,
        "%"
    )

    print(
        "Return/DD:",
        result[
            "return_to_dd"
        ]
    )


holdout_results = pd.DataFrame(
    holdout_results_list
)


print()
print("=" * 90)
print("HOLDOUT COMPARISON")
print("=" * 90)

print(

    holdout_results[
        [

            "model",

            "test_auc",

            "threshold",

            "trades",

            "direction_accuracy",

            "avg_return",

            "profit_factor",

            "growth",

            "max_dd",

            "return_to_dd",

        ]
    ]

    .to_string(
        index=False
    )

)


# ============================================================
# 27. FINAL DECISION
# ============================================================

print()
print("=" * 90)
print("FINAL MODEL TOURNAMENT DECISION")
print("=" * 90)


if (
    SELECTION_LEADER
    ==
    "RANDOM_FOREST"
):

    FINAL_DECISION = (

        "RANDOM FOREST REMAINS BASE-MODEL CHAMPION"

    )

    NEXT_STEP = (

        "Random Forestを維持し、"
        "次は特徴量側の改善または"
        "より異質なモデル群を検討します。"

    )

else:

    rf_holdout = (

        holdout_results.loc[

            holdout_results[
                "model"
            ]
            ==
            "RANDOM_FOREST"

        ]

        .iloc[0]

    )

    challenger_holdout = (

        holdout_results.loc[

            holdout_results[
                "model"
            ]
            ==
            SELECTION_LEADER

        ]

        .iloc[0]

    )

    tests = {

        "AUC":
            (
                challenger_holdout[
                    "test_auc"
                ]
                >
                rf_holdout[
                    "test_auc"
                ]
            ),

        "PF":
            (
                challenger_holdout[
                    "profit_factor"
                ]
                >
                rf_holdout[
                    "profit_factor"
                ]
            ),

        "AVG_RETURN":
            (
                challenger_holdout[
                    "avg_return"
                ]
                >
                rf_holdout[
                    "avg_return"
                ]
            ),

        "RETURN_DD":
            (
                challenger_holdout[
                    "return_to_dd"
                ]
                >
                rf_holdout[
                    "return_to_dd"
                ]
            ),

    }

    wins = sum(
        tests.values()
    )

    print(
        "Holdout metrics won:",
        wins,
        "/ 4"
    )

    for key, value in tests.items():

        print(
            key,
            ":",
            value
        )

    # ------------------------------------------
    # Rule fixed in advance:
    #
    # 4主要指標のうち3つ以上でRFを超え、
    # PF > 1
    # Avg Return > 0
    #
    # ならFull Champion Pipelineへ進める。
    #
    # まだRFを即置換はしない。
    # ------------------------------------------

    if (

        wins >= 3

        and

        challenger_holdout[
            "profit_factor"
        ]
        >
        1

        and

        challenger_holdout[
            "avg_return"
        ]
        >
        0

    ):

        FINAL_DECISION = (

            f"{SELECTION_LEADER} PASSES "
            "THE BASE-MODEL HOLDOUT"

        )

        NEXT_STEP = (

            f"{SELECTION_LEADER}を"
            "Calibration → Threshold → Session → "
            "30m Exit → Adaptive Position Sizing"
            "の完全Champion Pipelineへ組み込み、"
            "Random Forestと最終比較します。"

        )

    else:

        FINAL_DECISION = (

            "RANDOM FOREST REMAINS CHAMPION"

        )

        NEXT_STEP = (

            "Selection期間ではChallengerが強かったものの、"
            "2026 Holdoutで十分な再現性がありません。"
            "Random Forestを維持します。"

        )


print()
print(
    "RESULT:"
)

print(
    FINAL_DECISION
)

print()
print(
    "NEXT:"
)

print(
    NEXT_STEP
)


# ============================================================
# 28. NOISE / MODEL AGREEMENT DIAGNOSTIC
# ============================================================

print()
print("=" * 90)
print("MODEL DIAGNOSTIC")
print("=" * 90)


selection_model_auc_range = (

    selection_summary[
        "mean_auc"
    ].max()

    -

    selection_summary[
        "mean_auc"
    ].min()

)


best_mean_auc = (

    selection_summary[
        "mean_auc"
    ].max()

)


print(
    "Best Mean AUC:",
    best_mean_auc
)

print(
    "Mean AUC spread:",
    selection_model_auc_range
)


if (

    best_mean_auc
    <
    0.53

    and

    selection_model_auc_range
    <
    0.01

):

    MODEL_DIAGNOSIS = (

        "ALL MODELS ARE CLUSTERED NEAR THE SAME LOW AUC. "
        "FEATURE INFORMATION MAY BE THE BOTTLENECK."

    )

elif (

    selection_model_auc_range
    >=
    0.01

):

    MODEL_DIAGNOSIS = (

        "MODEL CHOICE APPEARS TO MATTER."

    )

else:

    MODEL_DIAGNOSIS = (

        "WEAK BUT NONZERO MODEL DIFFERENCES EXIST."

    )


print()
print(
    "Diagnosis:"
)

print(
    MODEL_DIAGNOSIS
)


# ============================================================
# 29. SAVE
# ============================================================

output_dir = (

    Path.cwd()

    /

    (
        "ml_model_tournament_"

        +

        datetime.now().strftime(
            "%Y%m%d_%H%M%S"
        )

    )

)


output_dir.mkdir(

    parents=True,

    exist_ok=False

)


selection_annual.to_csv(

    output_dir
    /
    "selection_annual_results.csv",

    index=False

)


selection_summary.to_csv(

    output_dir
    /
    "selection_summary.csv",

    index=False

)


selection_trades.to_csv(

    output_dir
    /
    "selection_oos_trades.csv"

)


bootstrap_results.to_csv(

    output_dir
    /
    "bootstrap_vs_random_forest.csv",

    index=False

)


holdout_results.to_csv(

    output_dir
    /
    "final_holdout_results.csv",

    index=False

)


if validation_search_frames:

    pd.concat(

        validation_search_frames,

        ignore_index=True

    ).to_csv(

        output_dir
        /
        "validation_threshold_search.csv",

        index=False

    )


summary_text = f"""
USDJPY ML MODEL TOURNAMENT
==========================

Features:
{len(FEATURES)}

Models:
{MODEL_NAMES}

Selection OOS years:
{SELECTION_YEARS}

Final Holdout:
{HOLDOUT_YEAR}

Selection Leader:
{SELECTION_LEADER}

Final Decision:
{FINAL_DECISION}

Next:
{NEXT_STEP}

Model diagnosis:
{MODEL_DIAGNOSIS}
"""


with open(

    output_dir
    /
    "tournament_summary.txt",

    "w",

    encoding="utf-8"

) as f:

    f.write(
        summary_text
    )


# ============================================================
# 30. FINISHED
# ============================================================

print()
print("=" * 90)
print("FINISHED")
print("=" * 90)

print(
    "Saved to:"
)

print(
    output_dir.resolve()
)

print()

print(
    "結果が出たら送ってほしい場所:"
)

print(
    "1. MODEL-SELECTION SUMMARY"
)

print(
    "2. PAIRED BLOCK BOOTSTRAP VS RANDOM FOREST"
)

print(
    "3. SELECTION-PERIOD WINNER"
)

print(
    "4. FINAL HOLDOUT"
)

print(
    "5. HOLDOUT COMPARISON"
)

print(
    "6. FINAL MODEL TOURNAMENT DECISION"
)

print(
    "7. MODEL DIAGNOSTIC"
)
